# Lab 2 - Text Pre-processing and Regular Expressions

Tasks for this lab: hashtags with regex, `re.compile()`, `re.split()`, and tokenization with spaCy vs NLTK.

Libraries used: nltk, spacy, pandas, kagglehub. Install them first with `pip install nltk spacy pandas kagglehub ipywidgets` if they are missing.

In [10]:
import warnings
warnings.filterwarnings("ignore")  # hide the tqdm progress bar warning

import glob
import os
import re
from collections import Counter

import kagglehub
import nltk
import pandas as pd
import spacy

# punkt is needed for the nltk tokenizers (punkt_tab on newer versions)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# download the small english model for spaCy if it is not installed
try:
    spacy.load("en_core_web_sm")
except OSError:
    spacy.cli.download("en_core_web_sm")

## Task 1: hashtags in the Apple tweets dataset

I need the total number of hashtags and the top 10. A hashtag is a `#` followed by letters, digits or underscore, so the pattern is `#\w+`. I lowercase everything first so `#Apple` and `#apple` are counted as one.

In [11]:
path = kagglehub.dataset_download("seriousran/appletwittersentimenttexts")
print(os.listdir(path))

csv_file = glob.glob(os.path.join(path, "*.csv"))[0]
df = pd.read_csv(csv_file)
print(df.shape)
df.head()

['apple-twitter-sentiment-texts.csv']
(1630, 2)


,text,sentiment
0,Wow. Yall needa step it up @Apple RT @heynyla:...,-1
1,What Happened To Apple Inc? http://t.co/FJEX...,0
2,Thank u @apple I can now compile all of the pi...,1
3,The oddly uplifting story of the Apple co-foun...,0
4,@apple can i exchange my iphone for a differen...,0


In [12]:
# the tweets are in the text column
text_col = "text" if "text" in df.columns else df.columns[0]

hashtag_pattern = re.compile(r"#\w+")

all_hashtags = []
for tweet in df[text_col].dropna():
    all_hashtags.extend(h.lower() for h in hashtag_pattern.findall(str(tweet)))

print("Total hashtags:", len(all_hashtags))
print("Unique hashtags:", len(set(all_hashtags)))

Total hashtags: 1616
Unique hashtags: 555


In [13]:
print("Top 10 hashtags:")
for tag, count in Counter(all_hashtags).most_common(10):
    print(tag, count)

Top 10 hashtags:
#aapl 400
#apple 162
#iphone 38
#december 37
#iphone6 29
#stocks 15
#ipad 14
#iphone6plus 13
#iphone6s 13
#tech 12


## Task 2: using re.compile()

In [14]:
text = "This year is 2021"

# pattern for one or more digits
digits = re.compile(r"\d+")

print(type(digits))

new_text = digits.sub("2022", text)
print(new_text)

<class 're.Pattern'>
This year is 2022


**What re.compile() does:** it turns the regex string into a pattern object. We can then use that object again and again (`sub`, `search`, `findall`...) without writing the pattern each time. It is useful when the same pattern is used many times, and it keeps the code cleaner.

## Task 3: using re.split()

In [15]:
text = "a 11 b 2 3 c 4"

parts = re.split(r"\d+", text)
print(parts)

['a ', ' b ', ' ', ' c ', '']


**What re.split() does:** it cuts the string every time the pattern matches and gives back the pieces in a list. It is like `str.split()` but the separator can be a regex. Here the digits are the separators, which is why there are spaces left in the pieces and an empty string at the end (the text ends with a digit).

## Task 4: tokenization with spaCy

In [16]:
text = "I'm enjoying the NLP course!"

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)

for token in doc:
    print(token.text)

I
'm
enjoying
the
NLP
course
!


**What spacy.load() does:** it loads a trained pipeline (here the small English model `en_core_web_sm`) and returns an `nlp` object. When we pass text to it, it runs the tokenizer and the other components (tagger, parser, etc.).

In [17]:
# same sentence with nltk
print(nltk.word_tokenize(text))

['I', "'m", 'enjoying', 'the', 'NLP', 'course', '!']


**Difference:** on this sentence both split `I'm` into `I` and `'m` and keep `!` as its own token, so the tokens are the same. The difference is the type: NLTK returns a plain list of strings, while spaCy returns `Token` objects that also have info like lemma, POS tag and `is_stop`.